# Clean HubAssessmentResults

Cleans a raw `HubAssessmentResults_yyyy-mm-dd.xlsx` export (Sheet1).

**Deviations / interpretations:**
- The notes say to read the `.xlsx` file (Sheet1), but then hand over a CSV/`escapechar`/backslash-protection read snippet identical to `Clean_HubDailyContent.ipynb`'s. That snippet doesn't apply here: `input/HubAssessmentResults_*.xlsx` is a genuine binary Excel file (verified ZIP magic bytes, not text), and `escapechar` is a `pd.read_csv`-only concept -- `pd.read_excel` reads cell values directly rather than tokenizing a delimited text stream, so there's no escape sequence to protect in the first place. There are also zero literal backslashes anywhere in the actual data. This notebook reads with a plain `pd.read_excel(..., sheet_name="Sheet1")`.
- `SectionName` contains genuine HTML tag/entity residue (e.g. `Physical wellness</p>`, `Stress&nbsp;</p>`) and the literal text `NULL` for `checkIn`-type rows (no section applies to a check-in). The notes' `strip_html()` function, applied once as given, fully cleans every case; `NULL` is left untouched, matching this repo's philosophy of preserving literal null-like strings rather than special-casing them.
- There is no duplicate-collapsing logic for this dataset, per the notes. The report below still counts duplicate rows in both the raw and cleaned data (via `df.duplicated()`) so any that occur are visible, without ever removing them -- `Duplicate rows collapsed` is always `0`.
- The output CSV is written with `encoding="utf-8"` explicitly, matching the other notebooks' reasoning (avoiding `cp1252` corruption of accented characters). `pd.read_excel` has no equivalent `encoding` argument to set -- Excel files aren't text-encoded the same way a CSV is; `openpyxl` handles Unicode internally.


## Imports

In [1]:
import csv
import html
import re
from pathlib import Path

import pandas as pd


## Schema constants

- `LS_COLS` -- the final column set and order for the cleaned output.
- `LS_STRING_COLS` -- the free-text columns that get whitespace-trimmed.
- `LS_INT_COLS` -- empty; this dataset has no numeric columns.
- `HTML_COLS` -- columns run through `strip_html()`.
- `SORT_COLS` -- the columns (and order) used for the final ascending sort.
- `FILENAME_RE` -- extracts the year/month from the input filename, used both to build `month_tag` and to auto-detect the input file.


In [2]:
LS_COLS = [
    "Date", "CompanyCode", "CompanyName", "CurrentCountry", "HomeCountry",
    "Operation", "AssessmentType", "SectionName", "Result",
]
LS_STRING_COLS = [
    "CompanyCode", "CompanyName", "CurrentCountry", "HomeCountry",
    "Operation", "AssessmentType", "SectionName", "Result",
]
LS_INT_COLS = []
HTML_COLS = ["SectionName"]
SORT_COLS = ["Date", "CompanyCode", "CompanyName", "CurrentCountry", "HomeCountry", "Operation"]

FILENAME_RE = re.compile(r"^HubAssessmentResults_(\d{4})-(\d{2})-\d{2}\.xlsx$")


## Locate the input file

`find_default_input` looks for a single `HubAssessmentResults_yyyy-mm-dd.xlsx` file in a given directory and returns it automatically. If none or several are found, it raises rather than silently guessing which one to use.


In [3]:
def find_default_input(directory: Path) -> Path:
    matches = sorted(p for p in directory.glob("HubAssessmentResults_*.xlsx") if FILENAME_RE.match(p.name))
    if not matches:
        raise FileNotFoundError(f"No HubAssessmentResults_yyyy-mm-dd.xlsx file found in {directory}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple candidate input files found in {directory}: "
            f"{[m.name for m in matches]}. Pass one explicitly."
        )
    return matches[0]


## Derive `month_tag` from the filename

The output name has the format `HubAssessmentResults_<month_tag>_cleaned.csv`, where `month_tag` is `yyyymm` for the month *before* the input filename's `yyyy-mm-dd` date suffix (the export date's month minus one), per the notes' worked example.


In [4]:
def month_tag_from_filename(path: Path) -> str:
    match = FILENAME_RE.match(path.name)
    if not match:
        raise ValueError(f"Filename '{path.name}' does not match expected pattern HubAssessmentResults_yyyy-mm-dd.xlsx")
    year, month = (int(g) for g in match.groups())
    # month_tag refers to the prior month's data, not the export date's month.
    year, month = (year - 1, 12) if month == 1 else (year, month - 1)
    return f"{year}{month:02d}"


## Strip HTML from `SectionName`

`SectionName` contains genuine HTML tag/entity residue left over from a rich-text field (e.g. trailing `</p>`, `&nbsp;`). Strips tags via regex, then unescapes any remaining HTML entities, per the notes.


In [5]:
HTML_TAG_RE = re.compile(r"<[^>]+>")


def strip_html(value: str) -> str:
    text = HTML_TAG_RE.sub("", value)
    text = html.unescape(text)
    return text.replace("\xa0", " ").strip()


## Cleaning logic

The core transformation, in the order implemented (the notes list these unordered):

1. Drop rows that are blank across every `LS_COLS` field.
2. Drop rows where `Date` is blank.
3. Reformat `Date` from its raw `yyyy-mm-dd hh:mm:ss.fffffff +00:00` shape to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
4. Strip HTML tags/entities from `SectionName`.
5. Lowercase `Result`.
6. Reorder/drop columns to match `LS_COLS`.
7. Trim and collapse whitespace on the `LS_STRING_COLS` fields.
8. Cast `LS_INT_COLS` to integer type (no-op -- this dataset has none).
9. Sort ascending by `Date`, `CompanyCode`, `CompanyName`, `CurrentCountry`, `HomeCountry`, `Operation`.


In [6]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    present_ls_cols = [c for c in LS_COLS if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # The raw timestamp carries 7-digit fractional seconds and a UTC offset (always
    # "+00:00"); %f zero-pads/truncates to 6-digit microseconds, and slicing off the
    # last 3 leaves milliseconds. Dropping the offset from the output format loses no
    # information since every row is already UTC.
    df["Date"] = (
        pd.to_datetime(df["Date"], format="%Y-%m-%d %H:%M:%S.%f %z")
        .dt.strftime("%Y-%m-%d %H:%M:%S.%f")
        .str[:-3]
    )

    for col in HTML_COLS:
        df[col] = df[col].apply(strip_html)

    df["Result"] = df["Result"].str.lower()

    df = df[[c for c in LS_COLS if c in df.columns]]

    for col in LS_STRING_COLS:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS, ascending=True).reset_index(drop=True)

    return df


## Configure the input file

Leave `INPUT_FILE` as `None` to auto-detect the single raw file in this notebook's `input/` folder, or set it to an explicit path to override (equivalent to the script's optional CLI argument).


In [7]:
NOTEBOOK_DIR = Path.cwd()
INPUT_FILE = None  # e.g. "input/HubAssessmentResults_2026-08-02.xlsx"

input_path = Path(INPUT_FILE).resolve() if INPUT_FILE else find_default_input(NOTEBOOK_DIR / "input")
month_tag = month_tag_from_filename(input_path)
input_path, month_tag


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubAssessmentResults_2026-08-02.xlsx'),
 '202607')

## Read the raw Excel file

Read everything as strings (`dtype=str`, `keep_default_na=False`) from `Sheet1`, so blank fields and literal `"NULL"` text pass through unchanged instead of being coerced or turned into `NaN`.

No CSV `escapechar`/backslash-protection handling here -- `pd.read_excel` reads cell values directly rather than tokenizing a delimited text stream, so there's no escape sequence to protect in the first place (see the note at the top of this notebook).


In [8]:
df_raw = pd.read_excel(input_path, sheet_name="Sheet1", dtype=str, keep_default_na=False)
df_raw.shape


(8960, 9)

## Apply the cleaning steps

In [9]:
df_cleaned = clean(df_raw)
df_cleaned.head()


,Date,CompanyCode,CompanyName,CurrentCountry,HomeCountry,Operation,AssessmentType,SectionName,Result
0,2026-07-01 00:21:23.916,AROLLAHUB,Arolla,France,France,Lyra France SASU,checkIn,NULL,sad
1,2026-07-01 00:24:03.369,FERRERO,Ferrero,United States of America,United States of America,Lyra Health International Ltd,checkIn,NULL,content
2,2026-07-01 00:46:11.677,SANDVIK,Sandvik Mining and Construction SEA Pte Ltd.,Hong Kong,Philippines,Lyra Health International Ltd,checkIn,NULL,happy
3,2026-07-01 01:06:12.517,BPWELLBEINGUSA,BP,United States of America,United States of America,Lyra Health International Ltd,checkIn,NULL,content
4,2026-07-01 01:43:03.624,KOCH,Koch,Mexico,Mexico,Lyra Health International Ltd,checkIn,NULL,happy


## Save the cleaned dataset

Written as `;`-delimited UTF-8 with minimal quoting, matching the other datasets' output convention. Saved to this notebook's `output/` folder.


In [10]:
output_dir = NOTEBOOK_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"HubAssessmentResults_{month_tag}_cleaned.csv"
df_cleaned.to_csv(output_path, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned)} rows -> {output_path}")


Cleaned 8960 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubAssessmentResults_202607_cleaned.csv


## Write summary report

Writes a plain-text report answering: which input file was read, the raw and cleaned row counts, how many duplicate rows exist in each of the raw and cleaned dataframes, the derived `month_tag`, how many rows were dropped for being blank / missing their key fields, and the output CSV's name, followed (after three blank lines) by `df_cleaned.describe()`. Saved to this notebook's `reports/` folder as `HubAssessmentResults_<month_tag>_report.txt`.

Per the notes, there is no duplicate-collapsing logic for this dataset -- `Duplicate rows collapsed` is always `0` below; the raw/cleaned duplicate counts exist purely to report on duplicates, not to justify removing them.


In [11]:
report_lines = [
    f"Input file: {input_path.name}",
    f"Raw row count: {len(df_raw)}",
    f"Raw duplicate rows: {int(df_raw.duplicated().sum())}",
    "=======================================================================",
    f"Month tag: {month_tag}",
    f"Blank/missing-key rows dropped: {len(df_raw) - len(df_cleaned)}",
    "Duplicate rows collapsed: 0",
    "=======================================================================",
    f"Cleaned row count: {len(df_cleaned)}",
    f"Cleaned duplicate rows: {int(df_cleaned.duplicated().sum())}",
    f"Output file: {output_path.name}",
]
report_text = "\n".join(report_lines) + "\n"
report_text += "\n\n\n" + df_cleaned.describe().to_string() + "\n"

reports_dir = NOTEBOOK_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / f"HubAssessmentResults_{month_tag}_report.txt"
report_path.write_text(report_text, encoding="utf-8")

print(report_text)
print(f"Report written -> {report_path}")


Input file: HubAssessmentResults_2026-08-02.xlsx
Raw row count: 8960
Raw duplicate rows: 0
Month tag: 202607
Blank/missing-key rows dropped: 0
Duplicate rows collapsed: 0
Cleaned row count: 8960
Cleaned duplicate rows: 0
Output file: HubAssessmentResults_202607_cleaned.csv



                           Date CompanyCode       CompanyName CurrentCountry   HomeCountry                      Operation AssessmentType SectionName   Result
count                      8960        8960              8960           8960          8960                           8960           8960        8960     8960
unique                     8959         826               825            114           117                             24              2           7       10
top     2026-07-02 11:47:59.916    MAFXLYRA  Majid Al Futtaim   South Africa  South Africa  Lyra Health International Ltd        checkIn        NULL  content
freq                          2         425               425           2876          2809 